In [1]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of united"
tokens = u_tokenizer.encode(prompt)
tokens.shape


torch.Size([7])

In [2]:
torch.manual_seed(1)
u_model = MasterModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, num_heads=2, context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)
sentence_meanings_with_attention_context

tensor([[ 1.4774, -1.2063,  0.2657, -0.5369],
        [-0.3433, -0.6376, -0.7323,  1.7132],
        [ 1.5882, -1.1775, -0.2286, -0.1821],
        [ 0.9593, -1.2064, -0.6988,  0.9458],
        [-1.4250,  0.8433, -0.4428,  1.0245],
        [ 1.5677, -0.9669,  0.1585, -0.7592],
        [ 0.9639, -1.2267, -0.7412,  1.0040]], grad_fn=<MulBackward0>)

In [21]:
sentence_meanings_with_attention_context.shape

torch.Size([7, 4])

In [6]:
u_model

MasterModel(
  (embedding): Embedding(64, 4)
  (pos_embedding): Embedding(32, 4)
  (self_attention): MultiHeadAttention(
    (multi_head_attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
    )
    (projection): Linear(in_features=4, out_features=4, bias=True)
  )
  (norm): LayerNorm()
)

In [23]:
u_model.embedding(tokens)

tensor([[-1.5256e+00, -7.5023e-01, -6.5398e-01, -1.6095e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.0017e-01, -6.0919e-01, -9.7977e-01, -1.6091e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-7.1214e-01,  3.0372e-01, -7.7731e-01, -2.5145e-01],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-2.2227e-01,  1.6871e+00,  2.2843e-01,  4.6764e-01]],
       grad_fn=<EmbeddingBackward0>)

In [24]:
tokens.unsqueeze(0)

tensor([[ 0, 61,  1, 61,  2, 61,  3]])

In [31]:
torch.manual_seed(1)

out_tokens = u_model(tokens)

In [33]:
out_tokens

tensor([[-0.5523, -0.4269, -0.6365,  0.3063],
        [ 0.0049, -0.2889, -0.1278, -0.2156],
        [-0.2110, -0.3828, -0.3719,  0.0068],
        [-0.2771, -0.1636, -0.2187, -0.1261],
        [-0.1539, -0.1642, -0.1840, -0.1723],
        [ 0.0203, -0.2972, -0.1447, -0.1711],
        [-0.0117, -0.2610, -0.1599, -0.1658]], grad_fn=<AddmmBackward0>)

In [9]:
#multihead, birden fazla baglamda inceleme. bu olmaz ise tek bi cumle icindeki baglama bakiyoruz.

In [10]:

import torch.nn as nn
from casual_self_attention import CasualSelfAttention

class DenemeMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, output_dim, context_length, num_heads, dropout_rate=0.5):
        super().__init__()
        assert embedding_dim % num_heads == 0

        head_dim = embedding_dim // num_heads
        self.heads = nn.ModuleList(
            [CasualSelfAttention(
                embedding_dim, head_dim, context_length, dropout_rate) for _ in range(num_heads)]
        )
        self.projection = nn.Linear(embedding_dim, output_dim)

    def forward(self, x):
        attention_outs = []
        for head in self.heads:
            head_out = head(x)
            attention_outs.append(head_out)
        concated_attention_outs = torch.concat(attention_outs, dim=1)
        return self.projection(concated_attention_outs)

deneme_multi_head_attention = DenemeMultiHeadAttention(4,4,32,2,dropout_rate=0)


In [11]:
out = deneme_multi_head_attention(torch.randn(4,4))
out.shape, out

(torch.Size([4, 4]),
 tensor([[ 0.1947,  0.3525,  0.0700, -0.5671],
         [ 0.0034,  0.3891, -0.5436, -0.7944],
         [-0.0472,  0.3895, -0.8001, -0.7882],
         [ 0.0614,  0.4009, -0.4116, -0.6936]], grad_fn=<AddmmBackward0>))

In [12]:
test = DenemeMultiHeadAttention(4,4,32,4,dropout_rate=0)
test_out = test(torch.randn(4,4))
test_out.shape, test_out

(torch.Size([4, 4]),
 tensor([[ 0.8449,  0.6277, -0.3168,  0.4635],
         [ 0.6821,  0.3840, -0.2924,  0.2257],
         [ 0.7033,  0.4051, -0.2700,  0.2922],
         [ 0.6552,  0.3525, -0.2244,  0.2590]], grad_fn=<AddmmBackward0>))

In [5]:
from norm_layer import LayerNorm
norm_layer = LayerNorm(4)
#norm_layer(out_tokens)
norm_layer(sentence_meanings_with_attention_context)

tensor([[ 1.4780, -1.2068,  0.2658, -0.5371],
        [-0.3435, -0.6380, -0.7327,  1.7141],
        [ 1.5893, -1.1784, -0.2287, -0.1822],
        [ 0.9896, -1.2445, -0.7208,  0.9758],
        [-1.4273,  0.8446, -0.4435,  1.0261],
        [ 1.5688, -0.9676,  0.1586, -0.7598],
        [ 0.9649, -1.2280, -0.7420,  1.0051]], grad_fn=<MulBackward0>)

In [14]:
nn.Parameter(torch.ones(4,3)) #matirsimiz

Parameter containing:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], requires_grad=True)

In [15]:
example_tensor = torch.tensor([[1,2,3], [4,5,6]], dtype=torch.float32)

mean = example_tensor.mean(dim=-1, keepdim=True)
var = example_tensor.var(dim=-1, keepdim=True, unbiased=False)

mean, var, example_tensor 


(tensor([[2.],
         [5.]]),
 tensor([[0.6667],
         [0.6667]]),
 tensor([[1., 2., 3.],
         [4., 5., 6.]]))

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

q_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 9469.88it/s]


In [9]:
q_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [ ]:
#mlp katmanini yapiyoruz multilayerperceptron, feed forward 
#weightleri egiticez, activation func vs ayarlicaz


![transformers-arch](https://deeprevision.github.io/posts/001-transformer/transformer.png)

![GELU-func](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*O5E-huBuY1UTHMmM--rhLQ.png)

In [14]:
import torch
import torch.nn as nn
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5*x*(1+torch.tanh(
            torch.sqrt(torch.tensor(2/torch.pi))*(x+0.044715*torch.pow(x,3))
            )
        )
gelu = GELU()
example_tensor = torch.tensor([[1,2,3],[4,5,6]], dtype=torch.float32)
gelu(example_tensor)


tensor([[0.8412, 1.9546, 2.9964],
        [3.9999, 5.0000, 6.0000]])

In [17]:
import torch.nn.functional as F
F.gelu(example_tensor, approximate="tanh")

tensor([[0.8412, 1.9546, 2.9964],
        [3.9999, 5.0000, 6.0000]])

In [ ]:
#feed neuron network
class MLP(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.gate_proj = nn.Linear(embedding_dim, hidden_dim)
        self.up_proj = nn.Linear(embedding_dim, hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, embedding_dim)
        #self.layers = nn.Sequential(
        #    nn.Linear(embedding_dim, hidden_dim),
        #    GELU(),
        #    nn.Linear(hidden_dim, embedding_dim)
        #)
        self.gelu = GELU()
    
    def forward(self, x):
        gate = self.gate_proj(x)
        gate = self.gelu(gate)
        up = self.up_proj(x)
        fuse = gate * up
        outputs = self.down_proj(fuse)
        return outputs